# Supplemental Tables for PepBind3D Manuscript

Generates four supplementary tables:

| Table | Content | Source |
|-------|---------|--------|
| **S1** | Deduplication decision logic | Hand-encoded from `clean_peplist()` in `IEDBTestPipeline.py` |
| **S2** | Per-allele Spearman correlations (IC50 and Kd) | Computed from `metadata.csv` |
| **S3** | Structural validation pairs | Loaded from `rmsd_per_pair.csv` |
| **S4** | Per-allele dataset composition | Derived from `metadata.csv` |

## Imports and paths

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
from pathlib import Path
import openpyxl

HF_DIR        = Path('/home/huntek1/main_project/data/IEDB_data_clean/huggingface')
METADATA_CSV  = HF_DIR / 'metadata.csv'
SCORES_DIR    = Path('/home/huntek1/main_project/data/IEDB_data_clean/IEDB_validation/scores_out')
OUT_01        = Path('/home/huntek1/main_project/data/IEDB_data_clean/IEDB_validation/01_structural_regen')
RMSD_CSV      = OUT_01 / 'rmsd_per_pair.csv'
OUT_DIR       = Path('/home/huntek1/main_project/data/IEDB_data_clean/IEDB_validation/supplemental_tables')
OUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', None)

## Table S1 — Curation pipeline decision logic

In [2]:
s1_rows = [
    {'Step': 'Pre-filter', 'Case': 'Missing quantitative measurement or assay units',
     'Condition': '`Quantitative_Measurement` or `Assay_Units` is NaN',
     'Action': 'Drop record'},
    {'Step': 'Pre-filter', 'Case': 'Variant Kd assay labels',
     'Condition': '`Assay_Response_Measured` ∈ {dissociation constant KD (~EC50), dissociation constant KD, dissociation constant (~IC50)}',
     'Action': 'Normalize to dissociation constant (KD) prior to filtering'},
    {'Step': 'Pre-filter', 'Case': 'Non-binding-affinity assay',
     'Condition': '`Assay_Response_Standardized` not in {dissociation constant (KD), half maximal inhibitory concentration (IC50)}',
     'Action': 'Drop record'},

    {'Step': 'Dedup (n=1)', 'Case': 'Single record',
     'Condition': '—', 'Action': 'Keep as-is'},

    {'Step': 'Dedup (n=2)', 'Case': 'PubMed provenance asymmetric',
     'Condition': 'One record has a PubMed ID, the other does not',
     'Action': 'Drop the record without PubMed ID'},
    {'Step': 'Dedup (n=2)', 'Case': 'Both lack PubMed, values agree',
     'Condition': 'Both records lack PubMed ID AND |Δvalue| ≤ 10 nM',
     'Action': 'Keep one representative (the first)'},
    {'Step': 'Dedup (n=2)', 'Case': 'Both lack PubMed, values disagree',
     'Condition': 'Both records lack PubMed ID AND |Δvalue| > 10 nM',
     'Action': 'Drop both records'},
    {'Step': 'Dedup (n=2)', 'Case': 'Both have PubMed, values agree',
     'Condition': 'Both records have PubMed ID AND |Δvalue| < 10 nM',
     'Action': 'Keep one representative'},
    {'Step': 'Dedup (n=2)', 'Case': 'Both have PubMed, values disagree',
     'Condition': 'Both records have PubMed ID AND |Δvalue| ≥ 10 nM',
     'Action': 'Flag for manual review against source publications'},

    {'Step': 'Dedup (n>2)', 'Case': 'Mixed PubMed status',
     'Condition': 'Some records have PubMed ID, others do not',
     'Action': 'Drop records without PubMed ID first, then re-evaluate'},
    {'Step': 'Dedup (n>2)', 'Case': 'All values within 10 nM',
     'Condition': 'All pairwise |Δvalue| ≤ 10 nM after PubMed filter',
     'Action': 'Keep one representative'},
    {'Step': 'Dedup (n>2)', 'Case': 'Values disagree by >10 nM',
     'Condition': 'At least one pair of records differs by >10 nM',
     'Action': 'Keep the value closest to the median'},

    {'Step': 'Post-filter', 'Case': 'Non-canonical amino acids',
     'Condition': 'Peptide sequence contains the "+" character',
     'Action': 'Exclude from structure generation'},
]

S1 = pd.DataFrame(s1_rows)
S1

,Step,Case,Condition,Action
0,Pre-filter,Missing quantitative measurement or assay units,`Quantitative_Measurement` or `Assay_Units` is NaN,Drop record
1,Pre-filter,Variant Kd assay labels,"`Assay_Response_Measured` ∈ {dissociation constant KD (~EC50), dissociation constant KD, dissociation constant (~IC50)}",Normalize to dissociation constant (KD) prior to filtering
2,Pre-filter,Non-binding-affinity assay,"`Assay_Response_Standardized` not in {dissociation constant (KD), half maximal inhibitory concentration (IC50)}",Drop record
3,Dedup (n=1),Single record,—,Keep as-is
4,Dedup (n=2),PubMed provenance asymmetric,"One record has a PubMed ID, the other does not",Drop the record without PubMed ID
5,Dedup (n=2),"Both lack PubMed, values agree",Both records lack PubMed ID AND |Δvalue| ≤ 10 nM,Keep one representative (the first)
6,Dedup (n=2),"Both lack PubMed, values disagree",Both records lack PubMed ID AND |Δvalue| > 10 nM,Drop both records
7,Dedup (n=2),"Both have PubMed, values agree",Both records have PubMed ID AND |Δvalue| < 10 nM,Keep one representative
8,Dedup (n=2),"Both have PubMed, values disagree",Both records have PubMed ID AND |Δvalue| ≥ 10 nM,Flag for manual review against source publications
9,Dedup (n>2),Mixed PubMed status,"Some records have PubMed ID, others do not","Drop records without PubMed ID first, then re-evaluate"


## Table S2 — Record attrition funnel

In [3]:
# Record attrition at each IEDB filtering stage (Rocco comment 5).
# Generated by attrition_counts.py; adjust the path if it wrote elsewhere.
ATTRITION_CSV = SCORES_DIR.parent / 'supplementary_table_S1_attrition.csv'
S2 = pd.read_csv(ATTRITION_CSV)
S2 = S2.rename(columns={'Note': 'Filter applied', 'Removed': 'Records removed'})
S2 = S2[['Stage', 'Filter applied', 'Records remaining', 'Records removed']]
S2

,Stage,Filter applied,Records remaining,Records removed
0,0. Raw IEDB MHC ligand records,IEDB MHC ligand bulk download,4883585,NaN
1,1. HLA-A / HLA-B allele restriction,MHC allele name begins with HLA-A or HLA-B,1389508,3494077.0
2,2. Quantitative value + assay units present,Both a quantitative measurement value and assay units,145858,1243650.0
3,3. Retained assay response (KD or IC50),Assay response is KD or IC50 (variant KD labels normalized),80347,65511.0
4,"4-6. Deduplication, flagging, and residue filters (net)","Duplicate resolution, flagged-record removal, non-canonical (+) exclusion",49488,30859.0


## Table S3 — Per-allele Spearman correlations

In [4]:
df = pd.read_csv(METADATA_CSV)
print('Rows:', len(df))
df = df[~df['flagged'].astype(bool)].copy()
print('After dropping flagged:', len(df))

print('Shape:', df.shape)
print('Columns:', list(df.columns))
print('\nUnique measurement_type values:', df['measurement_type'].unique())

Rows: 49488
After dropping flagged: 49473
Shape: (49473, 26)
Columns: ['allele_iedb', 'allele', 'peptide', 'peptide_length', 'measurement_type', 'measurement_value', 'measurement_units', 'assay_method', 'assay_response', 'pubmed_id', 'parent_protein', 'protein_accession', 'source_organism', 'assay_pdb_id', 'flagged', 'has_structures', 'num_pdbs', 'I_sc_best', 'I_sc_mean', 'reweighted_sc_best', 'reweighted_sc_mean', 'total_score_best', 'total_score_mean', 'rosetta_best_score', 'rosetta_mean_score', 'pdb_dir']

Unique measurement_type values: ['IC50' 'Kd']


In [5]:
# Robust filter: case-insensitive substring match
type_lower = df['measurement_type'].astype(str).str.lower()
is_ic50 = type_lower.str.contains('ic50', na=False)
is_kd   = type_lower.str.contains(r'\bkd\b', na=False, regex=True) | type_lower.str.fullmatch('kd', na=False)

print(f'IC50 rows: {is_ic50.sum():,}    Kd rows: {is_kd.sum():,}')

ic50 = df[is_ic50].copy()
kd   = df[is_kd].copy()

IC50 rows: 13,622    Kd rows: 35,851


In [6]:
MIN_N = 10
IC50_CENSORED, IC50_FLOOR = {20000, 50000, 70000}, 70000
KD_CENSORED,   KD_FLOOR   = {5000, 10000, 20000},  20000
SCORE_COL = 'I_sc_best'

# Merge in the parsed scores (same as notebook 02 cell 5)
scores = pd.read_csv(SCORES_DIR / 'score_summary.csv')
df['allele_dir'] = df['allele'].map(
    lambda a: (a[4:] if a.startswith('HLA-') else a).replace('*','').replace(':',''))

# Drop any score columns from a previous run so re-running doesn't create _x/_y
score_cols = [c for c in scores.columns if c not in ('allele_dir', 'peptide')]
df = df.drop(columns=[c for c in score_cols if c in df.columns], errors='ignore')

before = len(df)
df = df.merge(scores, on=['allele_dir','peptide'], how='left', validate='many_to_one')
assert len(df) == before, 'merge changed row count'
print(f'rows without {SCORE_COL}: {df[SCORE_COL].isna().sum():,}')

# Rebuild the subsets after the merge so they carry the score columns
type_lower = df['measurement_type'].astype(str).str.lower()
is_ic50 = type_lower.str.contains('ic50', na=False)
is_kd   = type_lower.str.fullmatch('kd', na=False)
ic50 = df[is_ic50].copy()
kd   = df[is_kd].copy()
print(f'IC50 rows: {len(ic50):,}   Kd rows: {len(kd):,}')

def per_allele_spearman(df_subset, censored, floor, measurement_label):
    keep = ~(df_subset['measurement_value'].isin(censored)
             | (df_subset['measurement_value'] >= floor))
    df_q = df_subset[keep].copy()
    df_q['log_value'] = np.log10(df_q['measurement_value'])
    rows = []
    for allele, grp in df_q.groupby('allele'):
        if len(grp) < MIN_N:
            continue
        rho, p = spearmanr(grp[SCORE_COL], grp['log_value'])
        rows.append({
            'Allele': allele,
            'Measurement': measurement_label,
            'n': len(grp),
            'Spearman ρ': round(rho, 3),
            'p-value': p,
            'Significant (p<0.05)': 'yes' if p < 0.05 else 'no',
        })
    cols = ['Allele', 'Measurement', 'n', 'Spearman ρ', 'p-value', 'Significant (p<0.05)']
    return pd.DataFrame(rows, columns=cols)

S3_ic50 = per_allele_spearman(ic50, IC50_CENSORED, IC50_FLOOR, 'IC50')
S3_kd   = per_allele_spearman(kd,   KD_CENSORED,   KD_FLOOR,   'Kd')

print(f'S3_ic50: {len(S3_ic50)} alleles | S3_kd: {len(S3_kd)} alleles')

rows without I_sc_best: 0
IC50 rows: 13,622   Kd rows: 35,851
S3_ic50: 28 alleles | S3_kd: 25 alleles


In [7]:
# Report usable (post-censoring) totals for the manuscript
for label, sub, cens, floor in [('IC50', ic50, IC50_CENSORED, IC50_FLOOR),
                                ('Kd',   kd,   KD_CENSORED,   KD_FLOOR)]:
    keep = ~(sub['measurement_value'].isin(cens) | (sub['measurement_value'] >= floor))
    print(f'{label}: {keep.sum():,} usable of {len(sub):,} total')

IC50: 10,901 usable of 13,622 total
Kd: 9,517 usable of 35,851 total


In [8]:
for label, sub in [('IC50', S3_ic50), ('Kd', S3_kd)]:
    n_sig = (sub['p-value'] < 0.05).sum()
    print(f'{label:>4}: n_alleles={len(sub):>3}, '
          f'median ρ={sub["Spearman ρ"].median():+.3f}, '
          f'IQR [{sub["Spearman ρ"].quantile(0.25):+.2f}, {sub["Spearman ρ"].quantile(0.75):+.2f}], '
          f'{n_sig} significant at p<0.05')

IC50: n_alleles= 28, median ρ=+0.182, IQR [+0.10, +0.37], 17 significant at p<0.05
  Kd: n_alleles= 25, median ρ=+0.157, IQR [+0.08, +0.22], 16 significant at p<0.05


In [9]:
# Final S3 (per-allele Spearman) with formatted p-values for display
S3 = pd.concat([S3_ic50, S3_kd], ignore_index=True)
S3['p-value'] = S3['p-value'].apply(lambda x: '< 0.001' if x < 0.001 else f'{x:.3f}')
S3 = S3.sort_values(['Measurement', 'Allele']).reset_index(drop=True)
S3

,Allele,Measurement,n,Spearman ρ,p-value,Significant (p<0.05)
0,A*01:01,IC50,190,0.326,< 0.001,yes
1,A*02:01,IC50,5261,0.353,< 0.001,yes
2,A*03:01,IC50,423,0.109,0.025,yes
3,A*11:01,IC50,469,0.194,< 0.001,yes
4,A*23:01,IC50,29,0.588,< 0.001,yes
5,A*24:02,IC50,587,0.313,< 0.001,yes
6,A*26:01,IC50,17,0.703,0.002,yes
7,A*29:02,IC50,48,0.426,0.003,yes
8,A*30:01,IC50,10,0.018,0.960,no
9,A*30:02,IC50,31,0.493,0.005,yes


## Table S4 — Structural validation pairs

In [10]:
rmsd_df = pd.read_csv(RMSD_CSV)
print('Shape:', rmsd_df.shape)
print('Columns:', list(rmsd_df.columns))

Shape: (52, 40)
Columns: ['allele_iedb', 'allele', 'peptide', 'peptide_length', 'measurement_type', 'measurement_value', 'measurement_units', 'assay_method', 'assay_response', 'pubmed_id', 'parent_protein', 'protein_accession', 'source_organism', 'assay_pdb_id', 'flagged', 'has_structures', 'num_pdbs', 'rosetta_best_score', 'rosetta_mean_score', 'pdb_dir', 'matched_pdb_id', 'mhc_chain_id', 'peptide_chain_id', 'resolution_angstrom', 'rmsd_min_of_25', 'n_decoys', 'mhc_alignment_rmsd', 'rmsd_best_score', 'rmsd_top5_mean', 'best_score', 'rmsd_best_score_reweighted_sc', 'rmsd_top5_mean_reweighted_sc', 'best_score_reweighted_sc', 'rmsd_best_score_total_score', 'rmsd_top5_mean_total_score', 'best_score_total_score', 'template_pdb', 'template_peptide', 'template_identity', 'rmsd_template']


In [11]:
rename_map = {
    'allele':               'Allele',
    'peptide':               'Peptide',
    'peptide_length':        'Length',
    'matched_pdb_id':        'Experimental PDB',
    'template_pdb':          'Template PDB',
    'template_identity':     'Template identity (%)',
    'rmsd_template':          'RMSD to threaded Template (Å)',
    'rmsd_best_score':       'RMSD best-by-score (Å)',
    'rmsd_top5_mean':        'RMSD top-5 mean (Å)',
    'rmsd_min_of_25':        'RMSD best-overall (Å)',
}
keep_cols = [c for c in rename_map if c in rmsd_df.columns]
S4 = rmsd_df[keep_cols].rename(columns=rename_map).copy()

# Template identity to %: if stored as fraction ≤ 1, multiply by 100
if 'Template identity (%)' in S3.columns and S3['Template identity (%)'].dropna().max() <= 1.5:
    S3['Template identity (%)'] = S3['Template identity (%)'] * 100

# Round numeric columns
for c in ['RMSD best-by-score (Å)', 'RMSD top-5 mean (Å)', 'RMSD best-overall (Å)',
          'RMSD best-overall (Å)', 'Relaxed starting-model RMSD (Å)']:
    if c in S4.columns:
        S4[c] = S4[c].round(2)
if 'Template identity (%)' in S4.columns:
    S4['Template identity (%)'] = S4['Template identity (%)'].round(1)

# Sort by template identity descending (no-template pairs sink to bottom)
if 'Template identity (%)' in S3.columns:
    S4 = S4.sort_values('Template identity (%)', ascending=False, na_position='last').reset_index(drop=True)

print(f'{len(S4)} rows')
S4

52 rows


,Allele,Peptide,Length,Experimental PDB,Template PDB,Template identity (%),RMSD to threaded Template (Å),RMSD best-by-score (Å),RMSD top-5 mean (Å),RMSD best-overall (Å)
0,A*02:01,GLCTLVAML,9,3MRE,3GSX,33.3,1.318564,1.38,1.35,1.02
1,A*02:01,VLHDDLLEA,9,3D25,3FT4,88.9,0.599518,1.01,0.98,0.89
2,A*02:01,CINGVCWTV,9,3MRG,3MRJ,88.9,1.004433,0.98,1.04,0.76
3,A*02:01,GMSRIGMEV,9,7KGP,6R2L,33.3,1.069974,1.05,1.10,1.05
4,A*02:01,GILGFVFTL,9,1OGA,5HHQ,88.9,1.160195,1.54,1.39,0.81
5,A*11:01,KTFPPTEPK,9,1X7Q,1Q94,22.2,0.947575,0.90,0.93,0.86
6,A*02:01,NLVPMVATV,9,6Q3K,3GSR,88.9,1.056343,1.18,1.14,1.03
7,B*27:05,KRWIILGLNK,10,4G9D,4G8I,90.0,0.928789,1.40,1.21,0.94
8,A*02:01,SLYNTVATL,9,2V2W,5NMH,88.9,1.129118,0.99,1.22,0.91
9,B*07:02,RPPIFIRRL,9,5WMO,4U1K,33.3,2.744241,2.91,2.98,2.50


## Table S5 — Per-allele dataset composition

In [12]:
# S4 describes the full released dataset, INCLUDING the 15 flagged records
# (they are in the release). Correlation tables (S2) exclude them.
df_all = pd.read_csv(METADATA_CSV)
tl = df_all['measurement_type'].astype(str).str.lower()
all_is_ic50 = tl.str.contains('ic50', na=False)
all_is_kd   = tl.str.fullmatch('kd', na=False)

comp_rows = []
for allele, grp in df_all.groupby('allele'):
    n_pep   = grp['peptide'].nunique()
    n_pairs = grp[['allele', 'peptide']].drop_duplicates().shape[0]
    n_ic50  = all_is_ic50.loc[grp.index].sum()
    n_kd    = all_is_kd.loc[grp.index].sum()
    lens    = grp['peptide'].dropna().str.len()
    comp_rows.append({
        'Allele': allele,
        'Unique peptides':   int(n_pep),
        'Peptide-HLA pairs': int(n_pairs),
        'IC50 measurements': int(n_ic50),
        'Kd measurements':   int(n_kd),
        'Length range':      f'{lens.min()}-{lens.max()}' if len(lens) else '—',
    })

S5 = pd.DataFrame(comp_rows).sort_values('Peptide-HLA pairs', ascending=False).reset_index(drop=True)
S5['% of dataset'] = (S5['Peptide-HLA pairs'] / S5['Peptide-HLA pairs'].sum() * 100).round(2)

pair_total = S5['Peptide-HLA pairs'].sum()
meas_total = S5['IC50 measurements'].sum() + S5['Kd measurements'].sum()
print(f'S4 pair total:        {pair_total:,}  (abstract says 49,268)')
print(f'S4 measurement total: {meas_total:,}  (Data Records says 49,488)')
assert pair_total == 49268, f'pair total {pair_total} != 49268 — reconcile with manuscript'
S5

S4 pair total:        49,268  (abstract says 49,268)
S4 measurement total: 49,488  (Data Records says 49,488)


,Allele,Unique peptides,Peptide-HLA pairs,IC50 measurements,Kd measurements,Length range,% of dataset
0,A*02:01,9102,9102,6195,3039,7-15,18.47
1,A*68:02,3457,3457,2451,1006,8-11,7.02
2,A*03:01,3097,3097,591,2516,8-12,6.29
3,B*15:01,2802,2802,283,2535,8-14,5.69
4,A*11:01,2508,2508,667,1846,8-11,5.09
5,B*07:02,2441,2441,561,1904,8-12,4.95
6,A*01:01,2313,2313,299,2031,8-12,4.69
7,B*58:01,2078,2078,39,2039,8-11,4.22
8,A*26:01,1994,1994,29,1965,9-11,4.05
9,B*57:01,1954,1954,90,1864,9-14,3.97


In [13]:
S6 = pd.read_csv(SCORES_DIR / 'metric_comparison_pooled.csv')
S6 = S6.rename(columns={'assay':'Assay', 'metric':'Score', 'agg':'Metric',
                        'rho':'Spearman ρ'})
S6['Spearman ρ'] = S6['Spearman ρ'].round(3)
S6['p'] = S6['p'].map(lambda x: f'{x:.1e}')
print(S6.to_string(index=False))

Assay         Score Metric  Spearman ρ        p     n
 IC50          I_sc   best       0.274 5.6e-187 10901
 IC50          I_sc   mean       0.219 2.0e-118 10901
 IC50        pep_sc   best       0.110  5.8e-31 10901
 IC50        pep_sc   mean       0.031  1.1e-03 10901
 IC50 reweighted_sc   best       0.132  2.2e-43 10901
 IC50 reweighted_sc   mean       0.054  1.8e-08 10901
 IC50   total_score   best       0.029  2.6e-03 10901
 IC50   total_score   mean       0.010  2.8e-01 10901
   Kd          I_sc   best       0.167  1.0e-60  9517
   Kd          I_sc   mean       0.140  6.2e-43  9517
   Kd        pep_sc   best       0.114  5.6e-29  9517
   Kd        pep_sc   mean       0.074  4.3e-13  9517
   Kd reweighted_sc   best       0.123  2.5e-33  9517
   Kd reweighted_sc   mean       0.076  1.2e-13  9517
   Kd   total_score   best       0.063  9.8e-10  9517
   Kd   total_score   mean       0.043  3.1e-05  9517


## Save all tables to one .xlsx

One sheet per table — open in Excel, copy individual tables into Word.

In [14]:
out_xlsx = OUT_DIR / 'PepBind3D_supplemental_tables.xlsx'
with pd.ExcelWriter(out_xlsx, engine='openpyxl') as xw:
    S1.to_excel(xw, sheet_name='S1_curation_logic',      index=False)
    S2.to_excel(xw, sheet_name='S2_attrition',           index=False)
    S3.to_excel(xw, sheet_name='S3_per_allele_spearman', index=False)
    S4.to_excel(xw, sheet_name='S4_validation_pairs',    index=False)
    S5.to_excel(xw, sheet_name='S5_allele_composition',  index=False)
    S6.to_excel(xw, sheet_name='S6_metric_comparison',   index=False)

print(f'Wrote: {out_xlsx}')
print('Sheets: S1-S6')

Wrote: /home/huntek1/main_project/data/IEDB_data_clean/IEDB_validation/supplemental_tables/PepBind3D_supplemental_tables.xlsx
Sheets: S1-S6
